# 03 — Validation-only net-payload feasibility and freeze

The candidate levels are predeclared as 0.05, 0.10, 0.20, 0.30 accounted net bpp. Feasibility is assessed on the **validation split only** using the blockwise side-information model. If the predeclared grid is not feasible for at least the configured fraction of validation images, a lower four-level grid is derived from the validation lower-tail capacity. The test split is not inspected here.

In [1]:
from pathlib import Path
import json, pandas as pd, numpy as np, yaml
from rdhlab.io import read_gray
from rdhlab.blockcodec import max_accounted_net_capacity
from rdhlab.pipeline import choose_payload_levels, manifest_sha256, save_payload_freeze

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
manifest_path=Path(config['dataset']['prepared_manifest'])
manifest=pd.read_csv(manifest_path)
val=manifest[manifest.split=='validation'].copy().reset_index(drop=True)
assert len(val)==2000
bs=int(config['dataset']['block_size'])
out=Path('/workspace/results/payload_freeze'); out.mkdir(parents=True,exist_ok=True)
cap_csv=out/'validation_capacity.csv'
existing=pd.read_csv(cap_csv) if cap_csv.exists() else pd.DataFrame()
done=set(existing.source_id.astype(str)) if len(existing) else set()
print('block size:',bs,'validation images:',len(val),'remaining:',len(val)-len(done))

block size: 64 validation images: 2000 remaining: 2000


In [2]:
for j,row in val.iterrows():
    sid=str(row.source_id)
    if sid in done: continue
    img=read_gray(row.path)
    r=max_accounted_net_capacity(img,bs)
    r.update({'source_id':sid,'pixels':img.size})
    pd.DataFrame([r]).to_csv(cap_csv,mode='a',header=not cap_csv.exists(),index=False)
    if (j+1)%100==0: print(f'{j+1}/2000')
cap=pd.read_csv(cap_csv)
assert len(cap)==2000
display(cap['max_net_bpp'].describe(percentiles=[.01,.05,.10,.25,.5,.75,.9,.95,.99]))

100/2000
200/2000
300/2000
400/2000
500/2000
600/2000
700/2000
800/2000
900/2000
1000/2000
1100/2000
1200/2000
1300/2000
1400/2000
1500/2000
1600/2000
1700/2000
1800/2000
1900/2000
2000/2000


count    2000.000000
mean        0.075173
std         0.073055
min         0.001343
1%          0.006789
5%          0.012888
10%         0.017699
25%         0.030209
50%         0.053146
75%         0.095562
90%         0.153506
95%         0.209460
99%         0.350271
max         0.756592
Name: max_net_bpp, dtype: float64

In [3]:
p=config['payload']
selection=choose_payload_levels(
    cap.max_net_bpp.to_numpy(),
    list(map(float,p['candidate_net_bpp'])),
    float(p['minimum_validation_feasibility']),
    int(p['fallback_levels']),
    float(p['fallback_quantile']),
    float(p['minimum_level_bpp']),
)
freeze={
    'experiment_version':config['project']['version'],
    'manifest_sha256':manifest_sha256(manifest_path),
    'split_used_for_selection':'validation',
    'block_size':bs,
    'minimum_validation_feasibility':float(p['minimum_validation_feasibility']),
    **selection,
}
freeze_path=Path('/workspace/config/frozen_payloads.json')
if freeze_path.exists():
    old=json.loads(freeze_path.read_text())
    assert old['manifest_sha256']==freeze['manifest_sha256'], 'Existing freeze belongs to a different manifest.'
    print('Freeze already exists; leaving it unchanged:',old)
else:
    save_payload_freeze(freeze_path,freeze)
    print('FROZEN:',freeze)
print('Freeze file:',freeze_path)

FROZEN: {'experiment_version': '0.2.0', 'manifest_sha256': '2ea33fc686971261b78184838853ad6313e660d566faf929e247b5251c9ab65f', 'split_used_for_selection': 'validation', 'block_size': 64, 'minimum_validation_feasibility': 0.95, 'levels': [0.003, 0.006, 0.009, 0.012], 'method': 'validation_q0.05_derived', 'candidate_feasibility': {0.05: 0.5295, 0.1: 0.2305, 0.2: 0.0565, 0.3: 0.0185}}
Freeze file: /workspace/config/frozen_payloads.json


Once `frozen_payloads.json` exists, **do not modify it after opening notebook 06 on the test split**. A different payload grid is a new experiment version.